# Benchmark: BGE-M3 Baseline vs Main FusionEncoder (Test Split)

Đánh giá và so sánh 2 hệ retrieval trên split test:
- Baseline: document lấy từ precomputed BGE-M3 trong features/bgem3, query encode bằng BGE-M3.
- Main model: FusionEncoder (image + text branches) với query encode BGE-M3.

Metrics: R@1, R@5, R@10, MdR, MnR, SumR

In [1]:
import json
import sys
import os
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer

cwd = Path.cwd().resolve()
if (cwd / 'source').exists():
    BASE_DIR = cwd
elif (cwd.parent / 'source').exists():
    BASE_DIR = cwd.parent
else:
    raise RuntimeError(f'Cannot locate project root from cwd={cwd}')

SOURCE_DIR = BASE_DIR / 'source'
if str(SOURCE_DIR) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIR))

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('BASE_DIR:', BASE_DIR)
print('DEVICE  :', DEVICE)
if torch.cuda.is_available():
    print('GPU     :', torch.cuda.get_device_name(0))

/home/urlab/miniconda3/envs/uav_ai/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


BASE_DIR: /media/urlab/KINGSTON/aic
DEVICE  : cuda
GPU     : NVIDIA GeForce RTX 5060 Ti


In [2]:
MAIN_CKPT = SOURCE_DIR / 'fusion_output' / 'best_fusion_model.pth'
BGEM3_PATH = BASE_DIR / 'features' / 'weights' / 'bgem3'
OUTPUT_JSON = SOURCE_DIR / 'benchmark_main_test_results.json'

CFG = {
    'base_dir': BASE_DIR,
    'bgem3_feature_dir': BASE_DIR / 'features' / 'bgem3',
    'dim': 1024,
    'vis_dim': 1152,
    'n_layers': 2,
    'n_heads': 16,
    'n_kv_heads': 4,
    'max_frames': 25,
    'max_segments': 15,
    'max_seg_tokens': 128,
    'batch_size': 4,
    'num_workers': os.cpu_count() or 4,
    'dual_softmax_tau': 0.01,
    'eval_query_max_length': 256,
    'eval_query_batch_size': 16,
}

print('Main model ckpt :', MAIN_CKPT)
print('BGE-M3 path     :', BGEM3_PATH)
print('BGE-M3 features :', CFG['bgem3_feature_dir'])
print('Output JSON     :', OUTPUT_JSON)

Main model ckpt : /media/urlab/KINGSTON/aic/source/fusion_output/best_fusion_model.pth
BGE-M3 path     : /media/urlab/KINGSTON/aic/features/weights/bgem3
BGE-M3 features : /media/urlab/KINGSTON/aic/features/bgem3
Output JSON     : /media/urlab/KINGSTON/aic/source/benchmark_main_test_results.json


In [3]:
from dataset import get_dataloader
from model import FusionEncoder

print('Imported get_dataloader and FusionEncoder successfully')

Imported get_dataloader and FusionEncoder successfully


In [4]:
test_loader = get_dataloader(
    split='test',
    base_dir=CFG['base_dir'],
    batch_size=CFG['batch_size'],
    num_workers=CFG['num_workers'],
    max_frames=CFG['max_frames'],
    max_segments=CFG['max_segments'],
    max_seg_tokens=CFG['max_seg_tokens'],
)

main_model = FusionEncoder(
    dim=CFG['dim'],
    vis_dim=CFG['vis_dim'],
    n_layers=CFG['n_layers'],
    n_heads=CFG['n_heads'],
    n_kv_heads=CFG['n_kv_heads'],
).to(DEVICE)

if not MAIN_CKPT.exists():
    raise FileNotFoundError(f'Main model checkpoint not found: {MAIN_CKPT}')

state = torch.load(MAIN_CKPT, map_location=DEVICE)
main_model.load_state_dict(state)
main_model.eval()
print('Main model loaded from:', MAIN_CKPT)

bgem3_tokenizer = AutoTokenizer.from_pretrained(str(BGEM3_PATH), local_files_only=True)
bgem3_model = AutoModel.from_pretrained(str(BGEM3_PATH), local_files_only=True)
bgem3_model.eval().to(DEVICE)
for p in bgem3_model.parameters():
    p.requires_grad = False

print('BGE-M3 loaded from:', BGEM3_PATH)
print('len(test_loader.dataset)=', len(test_loader.dataset))

Main model loaded from: /media/urlab/KINGSTON/aic/source/fusion_output/best_fusion_model.pth


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 3191.31it/s, Materializing param=pooler.dense.weight]                               


BGE-M3 loaded from: /media/urlab/KINGSTON/aic/features/weights/bgem3
len(test_loader.dataset)= 9222


## Evaluation Helpers (Baseline BGE-M3 and Main Model)

In [5]:
def encode_queries(
    texts,
    tokenizer,
    encoder_model,
    device,
    max_length=256,
    batch_size=16,
    show_progress=False,
    progress_desc='Encode queries',
):
    if len(texts) == 0:
        hidden_dim = int(getattr(encoder_model.config, 'hidden_size', 1024))
        return torch.empty((0, hidden_dim), device=device)

    if batch_size is None or batch_size <= 0:
        batch_size = len(texts)

    hidden_dim = int(getattr(encoder_model.config, 'hidden_size', 1024))
    ranges = range(0, len(texts), batch_size)
    if show_progress:
        total_batches = (len(texts) + batch_size - 1) // batch_size
        ranges = tqdm(ranges, total=total_batches, desc=progress_desc, leave=False, dynamic_ncols=True)

    all_embs = []
    with torch.inference_mode():
        for start in ranges:
            chunk = texts[start:start + batch_size]
            enc = tokenizer(
                chunk,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors='pt',
            ).to(device)

            if device.type == 'cuda':
                with torch.amp.autocast(device_type='cuda', enabled=True, dtype=torch.bfloat16):
                    out = encoder_model(**enc, return_dict=True)
                    h = out.last_hidden_state
            else:
                out = encoder_model(**enc, return_dict=True)
                h = out.last_hidden_state

            m = enc['attention_mask'].unsqueeze(-1).to(h.dtype)
            pooled = (h * m).sum(dim=1) / m.sum(dim=1).clamp_min(1e-6)
            all_embs.append(F.normalize(pooled.float(), p=2, dim=-1))

    if len(all_embs) == 0:
        return torch.empty((0, hidden_dim), device=device)
    return torch.cat(all_embs, dim=0)


def precompute_all_documents(loader, model, device=DEVICE):
    model.eval()
    doc_embs = []
    shot_ids = []

    pbar = tqdm(
        loader,
        total=max(1, len(loader)),
        desc='[test] Main docs',
        leave=False,
        dynamic_ncols=True,
    )

    with torch.inference_mode():
        for batch in pbar:
            with torch.amp.autocast(
                device_type='cuda' if device.type == 'cuda' else 'cpu',
                enabled=(device.type == 'cuda'),
                dtype=torch.bfloat16 if device.type == 'cuda' else torch.float32,
            ):
                e_plus, _, _ = model(
                    seg_tokens=batch['token_reprs'].to(device),
                    seg_pooled=batch['segment_pooled'].to(device),
                    visual_features=batch['visual_features'].to(device),
                    seg_timestamps=batch['seg_timestamps'].to(device),
                    frame_timestamps=batch['frame_timestamps'].to(device),
                    seg_mask=batch['segment_mask'].to(device),
                    visual_mask=batch['visual_mask'].to(device),
                    query_emb=None,
                    token_mask=batch['token_mask'].to(device),
                )

            doc_embs.append(e_plus.detach().cpu())
            shot_ids.extend(batch['shot_id'])
            pbar.set_postfix(docs=len(shot_ids))

    if len(doc_embs) == 0:
        return torch.empty((0, CFG['dim'])), []

    doc_embs = torch.cat(doc_embs, dim=0)
    return F.normalize(doc_embs, p=2, dim=-1), shot_ids


def load_queries_for_eval(split_dir: Path, query_field='positive'):
    q_items = []
    for jf in sorted(split_dir.glob('*.json')):
        video_id = jf.stem
        rows = json.loads(jf.read_text(encoding='utf-8'))
        grouped = {}
        for row in rows:
            if not isinstance(row, dict):
                continue
            sid = row.get('id')
            if sid is None or query_field not in row:
                continue
            sid = str(sid).zfill(3)
            q = str(row[query_field]).strip()
            if sid not in grouped and q:
                grouped[sid] = q
        for sid, q in grouped.items():
            q_items.append({'shot_id': f'{video_id}_{sid}', 'query': q})
    return q_items


def _collect_shot_pairs(split_dir: Path):
    pairs = []
    seen = set()
    for jf in sorted(split_dir.glob('*.json')):
        video_id = jf.stem
        rows = json.loads(jf.read_text(encoding='utf-8'))
        if not isinstance(rows, list):
            continue
        for row in rows:
            if not isinstance(row, dict):
                continue
            sid = row.get('id')
            if sid is None:
                continue
            sid = str(sid).zfill(3)
            key = (video_id, sid)
            if key in seen:
                continue
            seen.add(key)
            pairs.append(key)
    return pairs


def precompute_bgem3_baseline_documents(split_dir: Path):
    feat_root = CFG['bgem3_feature_dir']
    pairs = _collect_shot_pairs(split_dir)

    doc_embs = []
    shot_ids = []
    pbar = tqdm(pairs, total=max(1, len(pairs)), desc='[test] Baseline docs', leave=False, dynamic_ncols=True)
    for video_id, sid in pbar:
        pooled_npz = feat_root / video_id / f'{sid}_pooled.npz'
        if not pooled_npz.exists():
            continue

        data = np.load(pooled_npz)
        if 'pooled' not in data.files:
            continue

        pooled = data['pooled']
        if pooled.ndim == 1:
            vec = pooled.astype(np.float32)
        elif pooled.ndim == 2 and pooled.shape[0] > 0:
            vec = pooled.mean(axis=0).astype(np.float32)
        else:
            continue

        doc_embs.append(torch.from_numpy(vec))
        shot_ids.append(f'{video_id}_{sid}')
        pbar.set_postfix(docs=len(shot_ids))

    if len(doc_embs) == 0:
        return torch.empty((0, CFG['dim'])), []

    doc_embs = torch.stack(doc_embs, dim=0).float()
    return F.normalize(doc_embs, p=2, dim=-1), shot_ids


def dual_softmax(sim_matrix, tau=0.01):
    s = sim_matrix / tau
    s_row = torch.softmax(s, dim=1)
    s_col = torch.softmax(s, dim=0)
    return s_row * s_col


def compute_all_metrics(sim_matrix, gt_indices):
    sim_np = sim_matrix.detach().cpu().numpy()
    ranks = []
    for i, gt in enumerate(gt_indices):
        sorted_idx = np.argsort(-sim_np[i])
        rank = int(np.where(sorted_idx == gt)[0][0]) + 1
        ranks.append(rank)

    ranks = np.asarray(ranks)
    return {
        'R1': 100.0 * float(np.mean(ranks <= 1)),
        'R5': 100.0 * float(np.mean(ranks <= 5)),
        'R10': 100.0 * float(np.mean(ranks <= 10)),
        'MdR': float(np.median(ranks)),
        'MnR': float(np.mean(ranks)),
        'SumR': 100.0 * float(np.mean(ranks <= 1) + np.mean(ranks <= 5) + np.mean(ranks <= 10)),
    }


def evaluate_main_model(
    loader,
    model,
    split_dir: Path,
    query_field='positive',
    device=DEVICE,
    tau=None,
):
    tau = float(CFG['dual_softmax_tau'] if tau is None else tau)
    doc_embs, shot_ids = precompute_all_documents(loader, model, device=device)

    q_items = load_queries_for_eval(split_dir, query_field=query_field)
    shot_to_idx = {sid: i for i, sid in enumerate(shot_ids)}
    filtered_q = [x for x in q_items if x['shot_id'] in shot_to_idx]
    if len(filtered_q) == 0:
        raise RuntimeError('No overlap between GT queries and main-model docs on test split.')

    q_embs = encode_queries(
        [x['query'] for x in filtered_q],
        bgem3_tokenizer,
        bgem3_model,
        device,
        max_length=CFG['eval_query_max_length'],
        batch_size=CFG['eval_query_batch_size'],
        show_progress=True,
        progress_desc='[test] Main queries',
    )

    sim = torch.matmul(q_embs, doc_embs.to(device).T)
    sim_dsl = dual_softmax(sim, tau=tau)
    gt_indices = [shot_to_idx[x['shot_id']] for x in filtered_q]

    return compute_all_metrics(sim_dsl, gt_indices)


def evaluate_bgem3_baseline(
    split_dir: Path,
    query_field='positive',
    device=DEVICE,
    tau=None,
):
    tau = float(CFG['dual_softmax_tau'] if tau is None else tau)
    doc_embs, shot_ids = precompute_bgem3_baseline_documents(split_dir)
    if len(shot_ids) == 0:
        raise RuntimeError('No baseline BGE-M3 document embeddings found in features/bgem3.')

    q_items = load_queries_for_eval(split_dir, query_field=query_field)
    shot_to_idx = {sid: i for i, sid in enumerate(shot_ids)}
    filtered_q = [x for x in q_items if x['shot_id'] in shot_to_idx]
    if len(filtered_q) == 0:
        raise RuntimeError('No overlap between GT queries and baseline docs on test split.')

    q_embs = encode_queries(
        [x['query'] for x in filtered_q],
        bgem3_tokenizer,
        bgem3_model,
        device,
        max_length=CFG['eval_query_max_length'],
        batch_size=CFG['eval_query_batch_size'],
        show_progress=True,
        progress_desc='[test] Baseline queries',
    )

    sim = torch.matmul(q_embs, doc_embs.to(device).T)
    sim_dsl = dual_softmax(sim, tau=tau)
    gt_indices = [shot_to_idx[x['shot_id']] for x in filtered_q]

    return compute_all_metrics(sim_dsl, gt_indices)


print('Baseline + main evaluation helpers ready.')

Baseline + main evaluation helpers ready.


## Run Benchmark (test split only)

In [6]:
# Legacy helper block removed intentionally.
# Evaluation now uses evaluate_main_model() defined in the previous helper cell.
print('Using main-model benchmark helpers from the previous cell.')

Using main-model benchmark helpers from the previous cell.


## Run And Save Test Benchmark

In [7]:
results = {}
split_dir = BASE_DIR / 'data' / 'test'
query_field = 'positive'

print(f'[Baseline BGE-M3] test field={query_field}')
baseline_metrics = evaluate_bgem3_baseline(
    split_dir=split_dir,
    query_field=query_field,
    device=DEVICE,
)
results['baseline_bgem3/test'] = baseline_metrics
print(
    f"  R@1={baseline_metrics['R1']:.2f}  R@5={baseline_metrics['R5']:.2f}  "
    f"R@10={baseline_metrics['R10']:.2f}  MdR={baseline_metrics['MdR']:.1f}  "
    f"SumR={baseline_metrics['SumR']:.2f}"
)

print(f'\n[Main Model] test field={query_field}')
main_metrics = evaluate_main_model(
    loader=test_loader,
    model=main_model,
    split_dir=split_dir,
    query_field=query_field,
    device=DEVICE,
)
results['main/test'] = main_metrics
print(
    f"  R@1={main_metrics['R1']:.2f}  R@5={main_metrics['R5']:.2f}  "
    f"R@10={main_metrics['R10']:.2f}  MdR={main_metrics['MdR']:.1f}  "
    f"SumR={main_metrics['SumR']:.2f}"
)

comparison = {
    'delta_main_minus_baseline': {
        'R1': main_metrics['R1'] - baseline_metrics['R1'],
        'R5': main_metrics['R5'] - baseline_metrics['R5'],
        'R10': main_metrics['R10'] - baseline_metrics['R10'],
        'SumR': main_metrics['SumR'] - baseline_metrics['SumR'],
        'MdR': main_metrics['MdR'] - baseline_metrics['MdR'],
        'MnR': main_metrics['MnR'] - baseline_metrics['MnR'],
    }
}

print('\n[Compare] Main - Baseline')
print(
    f"  dR@1={comparison['delta_main_minus_baseline']['R1']:+.2f}  "
    f"dR@5={comparison['delta_main_minus_baseline']['R5']:+.2f}  "
    f"dR@10={comparison['delta_main_minus_baseline']['R10']:+.2f}  "
    f"dSumR={comparison['delta_main_minus_baseline']['SumR']:+.2f}"
)

out = {
    'timestamp': datetime.now(timezone.utc).isoformat().replace('+00:00', 'Z'),
    'split': 'test',
    'query_encoder': 'BGE-M3',
    'baseline': {
        'name': 'BGE-M3-precomputed-doc + BGE-M3-query',
        **baseline_metrics,
    },
    'main_model': {
        'name': 'SigLIP2+BGE-M3+FusionEncoder',
        'uses_image_branch_for_documents': True,
        **main_metrics,
    },
    'comparison': comparison,
    'config': {
        'max_frames': CFG['max_frames'],
        'max_segments': CFG['max_segments'],
        'max_seg_tokens': CFG['max_seg_tokens'],
        'dual_softmax_tau': CFG['dual_softmax_tau'],
        'eval_query_max_length': CFG['eval_query_max_length'],
        'eval_query_batch_size': CFG['eval_query_batch_size'],
    },
    'results': results,
}

OUTPUT_JSON.write_text(json.dumps(out, indent=2, ensure_ascii=False), encoding='utf-8')
print(f'Saved -> {OUTPUT_JSON}')

[Baseline BGE-M3] test field=positive


  R@1=51.62  R@5=73.77  R@10=80.38  MdR=1.0  SumR=205.77

[Main Model] test field=positive


  R@1=63.24  R@5=83.24  R@10=87.77  MdR=1.0  SumR=234.24

[Compare] Main - Baseline
  dR@1=+11.62  dR@5=+9.47  dR@10=+7.38  dSumR=+28.48
Saved -> /media/urlab/KINGSTON/aic/source/benchmark_main_test_results.json


In [8]:
header = f"{'Split':<12} {'Model':<30} {'R@1':>6} {'R@5':>6} {'R@10':>6} {'MdR':>6} {'SumR':>8}"
sep = '-' * len(header)
print(sep)
print(header)
print(sep)

m_base = results.get('baseline_bgem3/test')
m_main = results.get('main/test')
if m_base is None or m_main is None:
    raise RuntimeError('Missing benchmark result(s). Run the previous cell first.')

print(
    f"{'test':<12} {'BGE-M3 baseline':<30} "
    f"{m_base['R1']:>6.2f} {m_base['R5']:>6.2f} {m_base['R10']:>6.2f} {m_base['MdR']:>6.1f} {m_base['SumR']:>8.2f}"
)
print(
    f"{'test':<12} {'FusionEncoder(main)':<30} "
    f"{m_main['R1']:>6.2f} {m_main['R5']:>6.2f} {m_main['R10']:>6.2f} {m_main['MdR']:>6.1f} {m_main['SumR']:>8.2f}"
)
print(sep)

delta = out['comparison']['delta_main_minus_baseline']
print(
    f"{'test':<12} {'Delta(main-baseline)':<30} "
    f"{delta['R1']:>+6.2f} {delta['R5']:>+6.2f} {delta['R10']:>+6.2f} {delta['MdR']:>+6.1f} {delta['SumR']:>+8.2f}"
)
print(sep)

--------------------------------------------------------------------------------
Split        Model                             R@1    R@5   R@10    MdR     SumR
--------------------------------------------------------------------------------
test         BGE-M3 baseline                 51.62  73.77  80.38    1.0   205.77
test         FusionEncoder(main)             63.24  83.24  87.77    1.0   234.24
--------------------------------------------------------------------------------
test         Delta(main-baseline)           +11.62  +9.47  +7.38   +0.0   +28.48
--------------------------------------------------------------------------------
